# Phase 3 · V8 · which macro-F1 does the leaderboard compute? + per-document stray-label cleanup (no training, ~5 min)
**No training, no dev labels used for training.** Attach `phase2-backbones` + your probability dataset
(`p3_probs_V3A.pt`, `p3_probs_V3B.pt`, `p3_probs_V3C.pt`, and `p3_probs_V4A.pt` / `p3_probs_V4B.pt` if you have them).
1. Recomputes the dev score of every submission we already sent, under 4 versions of macro-F1 (pooled over sentences,
   over all 13 labels, mean of the 2 domains, mean over documents) and shows which version reproduces the leaderboard.
2. Cleans stray labels: in each document, a role predicted for fewer than k sentences is removed and that document is
   re-decoded with Viterbi (k chosen on dev under the matching metric).
3. Writes the zips. Submit `p3_v8_A` first, then `p3_v8_B`.

**Model (all Phase-3 notebooks):** each document's sentences are tokenised and packed, in order, into windows of up to
1,024 tokens, so every token also sees its neighbouring sentences → Phase-1 backbone (+ adaptation method) → **mean-pooled
sentence vectors** + relative position in the document + domain (CL/IT) → **document-level BiLSTM** → **role head** (13
labels) and **label-shift head** (does the role change from the previous sentence? auxiliary task, weight 0.5).
Loss: class-weighted cross-entropy (√ inverse frequency, for macro-F1) + label-shift loss. One model for both domains.
Model selection: best epoch by **macro-F1 on the official dev set** (CL + IT). Output: exactly one label per sentence,
order preserved, checked before zipping.

**Run:** GPU **T4 x2**, Internet **On**, Secret `capstone` (Hugging Face token), Add Input → **`phase2-backbones`**.
**Save & Run All.** Never submit a `smoke_*` zip.

In [ ]:
# 1 · Settings
import os, sys, json, glob, time, random, math, gc
import numpy as np, torch
BASE = dict(
    arch="arch4_mla", method="lorma", rank=16, alpha=32, bidirectional=True,
    bilstm=True, lsp=True, lsp_weight=0.5,      # document BiLSTM over sentences + label-shift auxiliary task (MTL)
    hidden=256, dropout=0.3, class_weighting="sqrt_inv", use_pos=True,
    window=1024, max_sent_tokens=128,           # sentences packed into 1024-token windows (neighbouring sentences = context)
    max_windows=4, eval_batch=16,               # training segment: 4 windows (~4k tokens) of one document per step
    epochs=20, patience=5, accum=4, lr=5e-4, lr_head=1e-3, weight_decay=0.01, warmup_frac=0.1, clip=1.0, seed=1337,
)
PLAN = []
OUT = "/kaggle/working"
SMOKE_CHECK = True
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
found = glob.glob("/kaggle/input/**/phase2-backbones/manifest.json", recursive=True) or \
        glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert found, "attach the dataset phase2-backbones (Add Input -> Your Datasets)"
ROOT = os.path.dirname(found[0])
print("backbones:", ROOT, "| GPUs:", torch.cuda.device_count(), "| runs:", [c["run"] for c in PLAN])

In [ ]:
# 2 · Hugging Face token (Kaggle secret "capstone" or "HF_TOKEN")
from huggingface_hub import login, whoami
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    for _n in ("capstone", "HF_TOKEN"):
        try:
            HF_TOKEN = UserSecretsClient().get_secret(_n)
            if HF_TOKEN: break
        except Exception:
            pass
except ImportError:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
assert HF_TOKEN, "add the Hugging Face token as a Kaggle secret and attach it"
login(token=HF_TOKEN); print("Hugging Face user:", whoami(token=HF_TOKEN)["name"])

## 3 · Code
`ner_model.py` (Phase-1 backbones, LoRA/LoRMA/adapters/BitFit, from Phase 2), `ner_data.py` (its helper), `rr.py` (Phase 3: data, rsLoRA, IA³, hierarchical multi-task model, training, distillation, metrics, submission).

In [ ]:
os.makedirs('p3src', exist_ok=True)

In [ ]:
%%writefile p3src/ner_data.py
# Phase 2 (L-NER) data utilities (pure Python, no torch):
#   gold parsing, words with exact character offsets, word-level BIO tags and span targets,
#   span decoding strategies, document-level label propagation, strict scoring, error analysis, submission.
import bisect
import re
from collections import Counter, defaultdict

# ======================================================================================================
# 1. Gold annotations
# ======================================================================================================
_START = ("start", "start_char", "start_offset", "begin", "s")
_END = ("end", "end_char", "end_offset", "stop", "e")
_LABEL = ("label", "type", "entity_type", "tag", "category")
_TEXT = ("text", "entity_text", "span", "mention", "value")
_IDS = ("id", "doc_id", "document_id", "docid")
_ENTS = ("entities", "annotations", "spans", "labels", "gold")


def _pick(d, keys):
    for k in keys:
        if k in d:
            return d[k]
    return None


def _entity(item):
    """one annotation in any common shape -> (start, end, label, text or None)"""
    if isinstance(item, (list, tuple)):
        s, e, lab = item[0], item[1], item[2]
        txt = item[3] if len(item) > 3 else None
    elif isinstance(item, dict):
        s, e, lab, txt = _pick(item, _START), _pick(item, _END), _pick(item, _LABEL), _pick(item, _TEXT)
    else:
        raise ValueError(f"unknown annotation format: {item!r}")
    return int(s), int(e), str(lab), (txt if isinstance(txt, str) else None)


def parse_gold(raw):
    """gold json (dict doc -> list, dict doc -> {entities: [...]}, or list of records) -> {doc: [(s,e,lab,txt)]}"""
    out = {}
    if isinstance(raw, list):
        for rec in raw:
            out[str(_pick(rec, _IDS))] = [_entity(x) for x in (_pick(rec, _ENTS) or [])]
        return out
    for doc, v in raw.items():
        if isinstance(v, dict):
            v = _pick(v, _ENTS) or []
        out[str(doc)] = [_entity(x) for x in v]
    return out


def parse_texts(raw):
    """text json -> {doc: text}"""
    if isinstance(raw, list):
        return {str(_pick(r, _IDS)): _pick(r, ("text", "content", "document")) for r in raw}
    return {str(k): (v if isinstance(v, str) else _pick(v, ("text", "content", "document"))) for k, v in raw.items()}


def end_convention(texts, gold):
    """Is `end` exclusive (text[s:e] == entity text) or inclusive (text[s:e+1])? Decided by the data."""
    exc = inc = total = 0
    bad = []
    for doc, ents in gold.items():
        t = texts.get(doc)
        if t is None:
            continue
        for s, e, lab, txt in ents:
            if txt is None:
                continue
            total += 1
            if t[s:e] == txt:
                exc += 1
            elif t[s:e + 1] == txt:
                inc += 1
            elif len(bad) < 5:
                bad.append((doc, s, e, lab, txt, t[s:e + 1]))
    conv = "inclusive" if inc > exc else "exclusive"
    return conv, dict(total_with_text=total, exclusive_match=exc, inclusive_match=inc, examples_no_match=bad)


def to_exclusive(gold, conv):
    """-> {doc: [(s, e_exclusive, label)]} (internal convention: python slice)"""
    add = 1 if conv == "inclusive" else 0
    return {d: [(s, e + add, lab) for s, e, lab, _ in ents] for d, ents in gold.items()}


# ======================================================================================================
# 2. Tokens and words with character offsets
# ======================================================================================================
# The pre-tokenisation decides where an entity can start or end. "bpe" = the split used in pretraining
# (a word keeps its punctuation: "Singh," is one piece). "fine" = punctuation is split off words, so
# "Singh" | "," can be cut exactly. Every piece is still encoded by the same BPE (same vocabulary).
SPLITS = {
    ("space", "bpe"): re.compile(r" ?\S+|\s+(?!\S)|\s+"),
    ("space", "fine"): re.compile(r" ?[^\W_]+| ?_+| ?[^\w\s]|\s+(?!\S)|\s+"),
    ("a1", "bpe"): re.compile(r"\S+|\s+"),
    ("a1", "fine"): re.compile(r"[^\W_]+|_+|[^\w\s]|\s+"),
}


def tokenizer_kind(tok):
    return "space" if getattr(tok, "space_prefix", False) else "a1"


def encode_words(tok, text, split="fine"):
    """text -> ids, wtok, wcs, wce
    ids  : BPE token ids of the whole text (decode(ids) == text)
    words: every pre-token piece that is not pure whitespace. For word w:
           wtok[w] = index of its first non-whitespace token (the token that represents the word),
           [wcs[w], wce[w]) = its characters without surrounding whitespace."""
    pat = SPLITS[(tokenizer_kind(tok), split)]
    ids, wtok, wcs, wce = [], [], [], []
    covered = 0
    for m in pat.finditer(text):
        chunk, base = m.group(), m.start()
        assert base == covered, f"pre-tokeniser skipped characters at {covered}"
        covered = m.end()
        char_of_byte = []
        for ci, ch in enumerate(chunk):
            char_of_byte.extend([ci] * len(ch.encode("utf-8")))
        pos, first, ws, we = 0, None, None, None
        for t in tok.encode_chunk(chunk.encode("utf-8")):
            n = len(tok.itos[t])
            s, e = base + char_of_byte[pos], base + char_of_byte[pos + n - 1] + 1
            pos += n
            while s < e and text[s].isspace():
                s += 1
            while e > s and text[e - 1].isspace():
                e -= 1
            if s < e:
                if first is None:
                    first, ws = len(ids), s
                we = e
            ids.append(t)
        if first is not None:
            wtok.append(first); wcs.append(ws); wce.append(we)
    assert covered == len(text), "pre-tokeniser did not cover the whole text"
    return ids, wtok, wcs, wce


# ======================================================================================================
# 3. Targets: BIO tags per word, and (start word, end word, label) for the span head
# ======================================================================================================
IGNORE = -100


def tag_vocab(labels):
    tags = ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]
    return tags, {t: i for i, t in enumerate(tags)}


def word_targets(wcs, wce, spans, tag2id, label2id):
    """gold char spans -> (BIO tag per word, [(first_word, last_word, label_id)], #overlapping spans skipped)"""
    n = len(wcs)
    tags, taken, wspans, conflicts = [0] * n, [False] * n, [], 0
    for s, e, lab in sorted(spans, key=lambda x: (x[0], -(x[1] - x[0]))):
        lo = bisect.bisect_right(wce, s)              # first word ending after s
        hi = bisect.bisect_left(wcs, e, lo)           # first word starting at/after e
        if lo >= hi:
            continue
        if any(taken[lo:hi]):
            conflicts += 1
            continue
        for k in range(lo, hi):
            tags[k] = tag2id[("B-" if k == lo else "I-") + lab]
            taken[k] = True
        wspans.append((lo, hi - 1, label2id[lab]))
    return tags, wspans, conflicts


def allowed_transitions(tags):
    """BIO validity: allowed[i][j] = may tag j follow tag i; start_ok[j] = may a sequence start with j."""
    K = len(tags)
    ok = [[True] * K for _ in range(K)]
    start_ok = [not t.startswith("I-") for t in tags]
    for i, a in enumerate(tags):
        for j, b in enumerate(tags):
            if b.startswith("I-") and not (a != "O" and a[2:] == b[2:]):
                ok[i][j] = False                      # I-X only after B-X or I-X
    return ok, start_ok


# ======================================================================================================
# 4. Decoding
# ======================================================================================================
def spans_from_tags(tags, wcs, wce, id2tag):
    """word tags -> [(s, e, label)]. B-X starts a span; I-X continues a span of type X; an I-X after O or
    after another type also starts a span (lenient repair, used with greedy argmax decoding)."""
    out, cur = [], None
    for t, s, e in zip(tags, wcs, wce):
        tag = id2tag[t]
        if tag == "O":
            if cur: out.append(cur); cur = None
            continue
        p, lab = tag.split("-", 1)
        if p == "B" or cur is None or cur[2] != lab:
            if cur: out.append(cur)
            cur = [s, e, lab]
        else:
            cur[1] = e
    if cur: out.append(cur)
    return [tuple(x) for x in out]


def spans_from_candidates(cands, wcs, wce, id2label):
    """span head: cands = {(i, j): (prob, label_id)} (label_id 0 = none) -> non-overlapping spans,
    highest probability first (greedy)."""
    used, out = set(), []
    for (i, j), (p, l) in sorted(cands.items(), key=lambda kv: -kv[1][0]):
        if l == 0 or any(k in used for k in range(i, j + 1)):
            continue
        used.update(range(i, j + 1))
        out.append((wcs[i], wce[j], id2label[l]))
    return sorted(out)


_WORDCH = re.compile(r"\w")


def propagate(spans, text, min_chars=4):
    """Document-level consistency: a string predicted as label L somewhere in the document is also
    labelled L at its other (unlabelled, whole-word) occurrences. Majority label per string."""
    votes = defaultdict(Counter)
    for s, e, l in spans:
        t = text[s:e]
        if len(t) >= min_chars and _WORDCH.search(t):
            votes[t][l] += 1
    covered = sorted((s, e) for s, e, _ in spans)
    starts = [s for s, _ in covered]

    def free(s, e):
        k = bisect.bisect_left(starts, e)
        return not any(cs < e and ce > s for cs, ce in covered[max(0, k - 50):k])

    out = list(spans)
    for t, c in votes.items():
        lab = c.most_common(1)[0][0]
        i = text.find(t)
        while i >= 0:
            j = i + len(t)
            boundary = (i == 0 or not text[i - 1].isalnum()) and (j == len(text) or not text[j].isalnum())
            if boundary and free(i, j):
                out.append((i, j, lab))
                k = bisect.bisect_left(starts, i); covered.insert(k, (i, j)); starts.insert(k, i)
            i = text.find(t, i + 1)
    return sorted(set(out))


def windows(n, size, stride):
    """window start positions covering [0, n); the last one is pulled back to end at n, so every window
    (except in documents shorter than size) has exactly `size` tokens - no padding needed."""
    if n <= size:
        return [0]
    starts = list(range(0, n - size + 1, stride))
    if starts[-1] + size < n:
        starts.append(n - size)
    return starts


# ======================================================================================================
# 5. Strict scoring and error analysis
# ======================================================================================================
def strict_scores(pred, gold, labels):
    """pred/gold: {doc: [(s,e,label)]} -> macro/micro P/R/F1 (exact start, end and label) + per label."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for doc in set(gold) | set(pred):
        g = set(map(tuple, gold.get(doc, []))); p = set(map(tuple, pred.get(doc, [])))
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        rows[l] = dict(P=P, R=R, F1=F, tp=tp[l], fp=fp[l], fn=fn[l], support=tp[l] + fn[l])
    used = [l for l in labels if rows[l]["support"] or rows[l]["fp"]]
    mac = lambda k: sum(rows[l][k] for l in used) / max(1, len(used))
    T, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    micro = 2 * T / (2 * T + FP + FN) if T else 0.0
    return dict(macro_F1=mac("F1"), macro_P=mac("P"), macro_R=mac("R"), micro_F1=micro, per_label=rows)


def error_breakdown(pred, gold):
    """every mistake classified: wrong label on an exact span, boundary error (overlap, same label),
    boundary+label error, missed entity (no overlapping prediction), spurious prediction."""
    c = Counter()
    for doc in set(gold) | set(pred):
        g = set(map(tuple, gold.get(doc, []))); p = set(map(tuple, pred.get(doc, [])))
        c["correct"] += len(p & g)
        ps, used = sorted(p - g), set()
        for s, e, l in sorted(g - p):
            over = [x for x in ps if x[0] < e and x[1] > s]
            if not over:
                c["missed"] += 1; continue
            used.update(over)
            if any(o[0] == s and o[1] == e for o in over): c["wrong_label_exact_span"] += 1
            elif any(o[2] == l for o in over): c["boundary_error_same_label"] += 1
            else: c["boundary_and_label_error"] += 1
        c["spurious"] += len([x for x in ps if x not in used])
    return dict(c)


def label_confusion(pred, gold):
    """exact-span matches with a different label: Counter[(gold label, predicted label)]"""
    c = Counter()
    for doc in gold:
        pm = {(s, e): l for s, e, l in pred.get(doc, [])}
        for s, e, l in gold[doc]:
            if (s, e) in pm and pm[(s, e)] != l:
                c[(l, pm[(s, e)])] += 1
    return c


# ======================================================================================================
# 6. Submission
# ======================================================================================================
def to_submission(pred, conv, texts=None):
    """{doc: [(s, e_exclusive, label)]} -> {doc: [[s, e, label(, text)]]} in the dataset's own end convention"""
    sub = 1 if conv == "inclusive" else 0
    out = {}
    for doc, spans in pred.items():
        rows = []
        for s, e, l in sorted(spans):
            row = [int(s), int(e - sub), l]
            if texts is not None: row.append(texts[doc][s:e])
            rows.append(row)
        out[doc] = rows
    return out

In [ ]:
%%writefile p3src/ner_model.py
# Phase 2 (L-NER) model side:
#   Phase-1 backbones as feature extractors (all 4 architectures, bidirectional or causal),
#   fine-tuning / parameter-efficient methods (full, partial, BitFit, LoRA, LoRMA, Houlsby/Pfeiffer adapters,
#   linear probe), prediction heads (linear softmax, CRF, span classifier), knowledge distillation,
#   sliding-window training and document-level prediction.
import bisect
import math
import os
import pickle
import random
import sys
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

import ner_data as nd

ARCH_INFO = {   # folder -> (tokenizer file, longest sequence the backbone supports)
    "arch1_decoder": ("a1_bpe8000.pkl", 1024),
    "arch2_encdec": ("a1_bpe8000.pkl", 768),
    "arch3_prefixlm": ("space_bpe8000.pkl", 1024),
    "arch4_mla": ("space_bpe8000.pkl", 1024),
}
NEG = -1e4


def autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available())


def _add_code_path(root):
    code = os.path.join(root, "code")
    if code not in sys.path:
        sys.path.insert(0, code)


def load_tokenizer(arch, root):
    _add_code_path(root)
    import a1_tokenizer, space_bpe  # noqa: F401  (classes needed to unpickle)
    return pickle.load(open(os.path.join(root, "tokenizers", ARCH_INFO[arch][0]), "rb"))


# ======================================================================================================
# 1. Backbones
# ======================================================================================================
def _mla_forward(self, x, cos, sin, cache=None):
    """MLA attention of architecture 4 without the causal mask (used when self.bidirectional is True)."""
    if cache is not None or not getattr(self, "bidirectional", False):
        return self._causal_forward(x, cos, sin, cache)
    from rope import apply_rope
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.h, self.nope + self.rope).transpose(1, 2)
    q_nope, q_rope = q.split([self.nope, self.rope], dim=-1)
    q_rope = apply_rope(q_rope, cos.to(q.dtype), sin.to(q.dtype))
    c_kv = self.kv_norm(self.kv_down(x))
    k_r = apply_rope(self.k_rope(x)[:, None], cos.to(q.dtype), sin.to(q.dtype))
    k_nope = self.k_up(c_kv).view(B, T, self.h, self.nope).transpose(1, 2)
    v = self.v_up(c_kv).view(B, T, self.h, self.vd).transpose(1, 2)
    q = torch.cat([q_nope, q_rope], dim=-1)
    k = torch.cat([k_nope, k_r.expand(B, self.h, T, self.rope)], dim=-1)
    out = F.scaled_dot_product_attention(q, k, v, is_causal=False, dropout_p=self.dropout if self.training else 0.0)
    return self.o_proj(out.transpose(1, 2).reshape(B, T, -1)), None


class Backbone(nn.Module):
    """Phase-1 language model reduced to a feature extractor: token ids (B,T) -> hidden states (B,T,d).
    bidirectional: drop the causal mask (arch 1, 4), use the whole window as prefix (arch 3); the arch-2
    encoder is bidirectional by design. keep_layers: keep only the first k blocks (distillation student).
    init="random": same architecture, random weights (control experiment for transfer from Phase 1)."""

    def __init__(self, arch, root, bidirectional=True, keep_layers=None, init="pretrained"):
        super().__init__()
        _add_code_path(root)
        self.arch, self.bidirectional = arch, bidirectional
        ck = torch.load(os.path.join(root, arch, "ckpt_best.pt"), map_location="cpu", weights_only=False)
        cfg = dict(ck["cfg"])
        if arch == "arch4_mla":
            from mla import MLAConfig, MLALM, MLAttention
            if not hasattr(MLAttention, "_causal_forward"):
                MLAttention._causal_forward = MLAttention.forward
                MLAttention.forward = _mla_forward
            m = MLALM(MLAConfig(**cfg))
        elif arch == "arch3_prefixlm":
            from prefixlm import PrefixLMConfig, PrefixLM
            m = PrefixLM(PrefixLMConfig(**cfg))
        elif arch == "arch2_encdec":
            from encdec import EncDecConfig, EncDecLM
            m = EncDecLM(EncDecConfig(**cfg))
        elif arch == "arch1_decoder":
            from model_arch import LMConfig, DecoderLM
            m = DecoderLM(LMConfig(**cfg))
        else:
            raise ValueError(arch)
        if init == "pretrained":
            missing, unexpected = m.load_state_dict(ck["model"], strict=False)
            assert not unexpected and all(x in ("rope", "pos_emb", "mask") or "rope" in x or "mask" in x
                                          or "pos_emb" in x for x in missing), (missing[:5], unexpected[:5])
        else:
            for mod in m.modules():
                if isinstance(mod, (nn.Linear, nn.Embedding)):
                    nn.init.normal_(mod.weight, std=0.02)
                    if getattr(mod, "bias", None) is not None: nn.init.zeros_(mod.bias)
        self.phase1_step, self.phase1_val = ck.get("step"), ck.get("val_loss")
        if arch == "arch2_encdec":                        # NER uses only the (bidirectional) encoder
            m.decoder = nn.ModuleList(); m.dec_norm = nn.Identity()
        if arch == "arch1_decoder":                       # A3 attention keeps its causal mask in a buffer
            for blk in m.blocks:
                blk.attn.causal_mask.fill_(True) if bidirectional else None
        if arch == "arch4_mla":
            for blk in m.blocks:
                blk.attn.bidirectional = bidirectional
        if keep_layers:
            blocks = self._blocks(m)
            del blocks[keep_layers:]
        m.head = None if arch != "arch2_encdec" else m.head   # LM head is not used (tied to the embedding)
        self.m, self.d, self.max_len = m, cfg["d_model"], ARCH_INFO[arch][1]

    @staticmethod
    def _blocks(m):
        return m.encoder if hasattr(m, "encoder") else m.blocks

    @property
    def blocks(self):
        return self._blocks(self.m)

    @property
    def final_norm(self):
        m = self.m
        return m.enc_norm if hasattr(m, "enc_norm") else (m.ln_f if hasattr(m, "ln_f") else m.norm)

    def forward(self, ids):
        m = self.m
        if self.arch == "arch4_mla":
            return m.hidden(ids)
        if self.arch == "arch3_prefixlm":
            return m.hidden(ids, prefix_len=ids.size(1) if self.bidirectional else 0)
        if self.arch == "arch2_encdec":
            return m.encode(ids)
        B, T = ids.shape                                  # arch 1: DecoderLM.forward without the LM head
        x = m.tok_emb(ids)
        if m.cfg.scale_emb:
            x = x * math.sqrt(m.cfg.d_model)
        if m.cfg.pos_emb_type == "sinusoidal":
            x = x + m.pos_emb[:T].unsqueeze(0)
        for blk in m.blocks:
            x = blk(x)
        return m.ln_f(x)


# ======================================================================================================
# 2. Fine-tuning / adaptation methods
# ======================================================================================================
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2021): W x + (alpha/r) * B A x, with W frozen, A random, B = 0 (starts at W)."""

    def __init__(self, base, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA (low-rank MULTIPLICATIVE adaptation): W' = (I + (alpha/r) B A) W, with W frozen.
    The identity term gives the product full rank ("rank inflation" by identity), and B = 0 makes the
    adapted layer start exactly at the pretrained one. Computed as h + s * B(A h) with h = W x, so the
    d_out x d_in product (BA)W is never formed. A bias, if any, is added after the multiplicative update."""

    def __init__(self, base, r=16, alpha=32):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        h = h + self.scale * F.linear(F.linear(h, self.A), self.B)
        return h if self.base.bias is None else h + self.base.bias


class Adapter(nn.Module):
    """bottleneck adapter (Houlsby et al. 2019): x + up(GELU(down(x))), up = 0 at start (identity)."""

    def __init__(self, d, bottleneck=64):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(F.gelu(self.down(x)))


class AdapterWrap(nn.Module):
    """applies an adapter to the output of a sub-layer (attention or feed-forward), before the residual add"""

    def __init__(self, module, d, bottleneck):
        super().__init__()
        self.module, self.adapter = module, Adapter(d, bottleneck)

    def __getattr__(self, name):                      # keep attributes of the wrapped module reachable
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self._modules["module"], name)

    def forward(self, *args, **kwargs):
        out = self.module(*args, **kwargs)
        if isinstance(out, tuple):
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)


def _replace_linears(root_module, make):
    n = 0
    for name, child in list(root_module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(root_module, name, make(child)); n += 1
        else:
            n += _replace_linears(child, make)
    return n


def apply_method(net, cfg):
    """Freeze/unfreeze and insert adaptation modules according to cfg["method"]. The task head is always
    trained. Returns a short description for the record."""
    bb, method = net.backbone, cfg["method"]
    for p in bb.parameters():
        p.requires_grad = method == "full_ft"
    desc = method
    if method == "partial":                           # layer-wise: only the top k blocks + final norm
        k = cfg.get("train_top_layers", 4)
        for blk in list(bb.blocks)[-k:]:
            for p in blk.parameters(): p.requires_grad = True
        for p in bb.final_norm.parameters(): p.requires_grad = True
        desc = f"partial (top {k} of {len(bb.blocks)} blocks)"
    elif method == "bitfit":                          # only bias terms (Ben Zaken et al. 2022)
        n = 0
        for name, p in bb.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad = True; n += p.numel()
        assert n > 0, "BitFit needs bias terms - this backbone has none (use arch 1 or arch 2)"
        desc = f"bitfit ({n:,} bias parameters)"
    elif method in ("lora", "lorma"):
        r, a = cfg.get("rank", 16), cfg.get("alpha", 32)
        make = (lambda lin: LoRALinear(lin, r, a, cfg.get("lora_dropout", 0.05))) if method == "lora" \
            else (lambda lin: LoRMALinear(lin, r, a))
        n = sum(_replace_linears(blk, make) for blk in bb.blocks)
        for mod in bb.modules():
            if isinstance(mod, (LoRALinear, LoRMALinear)):
                mod.A.requires_grad = True; mod.B.requires_grad = True
        desc = f"{method} (rank {r}, alpha {a}, {n} linear layers)"
    elif method in ("houlsby", "pfeiffer"):           # Houlsby: after attention and FFN; Pfeiffer: after FFN only
        bott = cfg.get("bottleneck", 64)
        for blk in bb.blocks:
            if method == "houlsby":
                blk.attn = AdapterWrap(blk.attn, bb.d, bott)
            blk.ffn = AdapterWrap(blk.ffn, bb.d, bott)
        for mod in bb.modules():
            if isinstance(mod, Adapter):
                for p in mod.parameters(): p.requires_grad = True
            if isinstance(mod, (nn.LayerNorm,)) or type(mod).__name__ == "RMSNorm":
                for p in mod.parameters(): p.requires_grad = True     # Houlsby also trains the norms
        desc = f"{method} adapters (bottleneck {bott})"
    elif method == "probe":                           # frozen backbone, only the head (linear probe)
        desc = "linear probe (frozen backbone)"
    elif method != "full_ft":
        raise ValueError(method)
    return desc


def count_params(net):
    total = sum(p.numel() for p in net.parameters())
    train = sum(p.numel() for p in net.parameters() if p.requires_grad)
    return total, train


# ======================================================================================================
# 3. Prediction heads
# ======================================================================================================
class CRF(nn.Module):
    """linear-chain CRF over word tags. Learned transitions + fixed BIO constraints (forbidden moves = -1e4)."""

    def __init__(self, tags):
        super().__init__()
        K = len(tags)
        ok, start_ok = nd.allowed_transitions(tags)
        self.trans = nn.Parameter(torch.zeros(K, K)); self.start = nn.Parameter(torch.zeros(K))
        self.end = nn.Parameter(torch.zeros(K))
        self.register_buffer("ban", (~torch.tensor(ok)).float() * NEG)
        self.register_buffer("ban_start", (~torch.tensor(start_ok)).float() * NEG)

    def scores(self, train=False):
        # training windows may start inside an entity (first word I-X), so the start constraint is only
        # applied when decoding whole documents
        return self.trans + self.ban, (self.start if train else self.start + self.ban_start), self.end

    def nll(self, em, tags, mask):
        """em (B,W,K) float, tags (B,W) (pads ignored), mask (B,W) bool, prefix-contiguous"""
        trans, start, end = self.scores(train=True)
        B, W, K = em.shape
        tags = tags.clamp(min=0)
        mf = mask.float()
        # score of the gold path
        gold = start[tags[:, 0]] + em[:, 0].gather(1, tags[:, :1]).squeeze(1)
        for t in range(1, W):
            step = trans[tags[:, t - 1], tags[:, t]] + em[:, t].gather(1, tags[:, t:t + 1]).squeeze(1)
            gold = gold + step * mf[:, t]
        last = tags.gather(1, (mask.sum(1) - 1).clamp(min=0)[:, None]).squeeze(1)
        gold = gold + end[last]
        # log partition
        alpha = start[None] + em[:, 0]
        for t in range(1, W):
            nxt = torch.logsumexp(alpha[:, :, None] + trans[None], dim=1) + em[:, t]
            alpha = torch.where(mask[:, t, None], nxt, alpha)
        logz = torch.logsumexp(alpha + end[None], dim=1)
        has = mask[:, 0]
        return ((logz - gold) * has.float()).sum() / has.float().sum().clamp(min=1)


def viterbi(em, trans, start, end):
    """em (n,K) -> best tag sequence (list) under transition scores (CPU)"""
    n, K = em.shape
    if n == 0:
        return []
    score = start + em[0]
    back = []
    for t in range(1, n):
        s = score[:, None] + trans
        best, idx = s.max(0)
        score = best + em[t]; back.append(idx)
    score = score + end
    tag = int(score.argmax())
    path = [tag]
    for idx in reversed(back):
        tag = int(idx[tag]); path.append(tag)
    return path[::-1]


class NERModel(nn.Module):
    """backbone -> word representations (first token of each word) -> head.
    head = "linear": softmax over BIO tags (cross-entropy, optional class weights)
           "crf":    BIO tag emissions + CRF (sequence-level likelihood)
           "span":   classify every word span up to max_span_words (start/end projections, bilinear term,
                     width embedding), label 0 = not an entity."""

    def __init__(self, backbone, tags, labels, head="linear", dropout=0.2, class_weights=None,
                 max_span_words=32, span_dim=256, neg_weight=0.5):
        super().__init__()
        self.backbone, self.head_type, self.tags = backbone, head, tags
        self.id2label = {i + 1: l for i, l in enumerate(labels)}          # span head: class 0 = no entity
        n_labels = len(labels)
        d, K = backbone.d, len(tags)
        self.drop = nn.Dropout(dropout)
        if head in ("linear", "crf"):
            self.head = nn.Linear(d, K)
            nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)
            self.register_buffer("class_weights", class_weights if class_weights is not None else torch.ones(K))
            if head == "crf":
                self.crf = CRF(tags)
        elif head == "span":
            C = n_labels + 1
            self.maxw = max_span_words
            self.head = nn.ModuleDict(dict(
                s=nn.Sequential(nn.Linear(d, span_dim), nn.GELU()), e=nn.Sequential(nn.Linear(d, span_dim), nn.GELU()),
                ls=nn.Linear(span_dim, C), le=nn.Linear(span_dim, C), bil=nn.Linear(span_dim, C, bias=False),
                width=nn.Embedding(max_span_words, C)))
            w = torch.ones(C); w[0] = neg_weight
            self.register_buffer("class_weights", w)
        else:
            raise ValueError(head)

    def word_states(self, ids, wpos):
        h = self.backbone(ids)                                            # (B,T,d)
        return h.gather(1, wpos[:, :, None].expand(-1, -1, h.size(-1)))   # (B,W,d)

    def span_logits(self, hw, wmask):
        H = self.head
        s, e = H["s"](hw), H["e"](hw)                                     # (B,W,p)
        B, W, p = s.shape
        k = torch.arange(self.maxw, device=hw.device)
        j = torch.arange(W, device=hw.device)[:, None] + k[None]          # (W,maxw)
        valid = (j < W)[None] & wmask[:, :, None] & wmask.gather(1, j.clamp(max=W - 1).reshape(1, -1).expand(B, -1)).view(B, W, -1)
        ej = e.gather(1, j.clamp(max=W - 1).reshape(1, -1, 1).expand(B, -1, p)).view(B, W, self.maxw, p)
        logits = H["ls"](s)[:, :, None] + H["le"](ej) + H["bil"](s[:, :, None] * ej) + H["width"](k)[None, None]
        return logits.float(), valid

    def forward(self, ids, wpos, wmask, wtags=None, stgt=None):
        with autocast():
            hw = self.drop(self.word_states(ids, wpos))
            if self.head_type == "span":
                logits, valid = self.span_logits(hw, wmask)
            else:
                logits = self.head(hw).float()
        if self.head_type == "span":
            if stgt is None:                                   # invalid slots -> certainly "no entity"
                inv = ~valid
                out = logits.masked_fill(inv[..., None], NEG)
                out[..., 0] = out[..., 0].masked_fill(inv, 0.0)
                return out
            tgt = stgt.masked_fill(~valid, nd.IGNORE)
            return F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1),
                                   weight=self.class_weights, ignore_index=nd.IGNORE).reshape(1)
        if wtags is None:
            return logits
        if self.head_type == "crf":
            return self.crf.nll(logits, wtags, wmask).reshape(1)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), wtags.reshape(-1),
                               weight=self.class_weights, ignore_index=nd.IGNORE).reshape(1)


class KDWrapper(nn.Module):
    """Knowledge distillation (Hinton et al. 2015) at word level:
    loss = a * CE(student, gold) + (1 - a) * T^2 * KL(teacher_T || student_T). The teacher is frozen."""

    def __init__(self, student, teacher, T=2.0, alpha=0.5):
        super().__init__()
        self.student, self.teacher, self.T, self.alpha = student, teacher, T, alpha
        for p in teacher.parameters(): p.requires_grad = False

    def train(self, mode=True):
        super().train(mode); self.teacher.eval(); return self

    def forward(self, ids, wpos, wmask, wtags=None, stgt=None):
        if wtags is None:
            return self.student(ids, wpos, wmask)
        with torch.no_grad():
            t = self.teacher(ids, wpos, wmask)                            # (B,W,K)
        s = self.student(ids, wpos, wmask)
        m = wmask.reshape(-1)
        ce = F.cross_entropy(s.reshape(-1, s.size(-1)), wtags.reshape(-1), ignore_index=nd.IGNORE)
        sl, tl = s.reshape(-1, s.size(-1))[m] / self.T, t.reshape(-1, t.size(-1))[m] / self.T
        kl = F.kl_div(F.log_softmax(sl, -1), F.softmax(tl, -1), reduction="batchmean") * self.T ** 2
        return (self.alpha * ce + (1 - self.alpha) * kl).reshape(1)


# ======================================================================================================
# 4. Documents -> windows -> batches
# ======================================================================================================
def prepare_docs(texts, gold, tok, tag2id, label2id, split="fine"):
    docs, conflicts = {}, 0
    for d, t in texts.items():
        ids, wtok, wcs, wce = nd.encode_words(tok, t, split)
        r = dict(ids=ids, wtok=wtok, wcs=wcs, wce=wce)
        if gold is not None:
            r["wtags"], r["wspans"], c = nd.word_targets(wcs, wce, gold.get(d, []), tag2id, label2id); conflicts += c
        docs[d] = r
    return docs, conflicts


class Windowing:
    """Cuts every document into windows of `size` tokens. With header_tokens > 0, the first tokens of the
    document (case title, parties, counsel) are put in front of every window, followed by a separator,
    so each window can see who the appellant/respondent/counsel are. Header words are never labelled."""

    def __init__(self, tok, size, header_tokens=0, max_len=1024):
        self.sep = tok.encode("\n\n") if header_tokens else []
        self.H = header_tokens
        self.size = min(size, max_len - (header_tokens + len(self.sep) if header_tokens else 0))
        assert self.size >= 64, "window too small"

    def windows(self, docs, stride):
        out = []
        for d, r in docs.items():
            if r["wtok"]:
                for s in nd.windows(len(r["ids"]), self.size, stride):
                    out.append((d, s, min(self.size, len(r["ids"]))))
        return out

    def window_words(self, r, s, L):
        w0 = bisect.bisect_left(r["wtok"], s); w1 = bisect.bisect_left(r["wtok"], s + L)
        return w0, w1

    def batch(self, docs, b, device, head=None, maxw=None):
        ids, wpos, wtags, ranges, stgts = [], [], [], [], []
        for d, s, L in b:
            r = docs[d]
            head_ids = (r["ids"][:self.H] + self.sep) if self.H else []
            off = len(head_ids) - s
            ids.append(head_ids + r["ids"][s:s + L])
            w0, w1 = self.window_words(r, s, L)
            ranges.append((w0, w1))
            wpos.append([r["wtok"][w] + off for w in range(w0, w1)])
            if "wtags" in r:
                wtags.append(r["wtags"][w0:w1])
        T = len(ids[0]); assert all(len(x) == T for x in ids)
        Wm = max(1, max(len(x) for x in wpos))
        P = torch.zeros(len(b), Wm, dtype=torch.long); M = torch.zeros(len(b), Wm, dtype=torch.bool)
        for i, x in enumerate(wpos):
            P[i, :len(x)] = torch.tensor(x, dtype=torch.long); M[i, :len(x)] = True
        out = dict(ids=torch.tensor(ids, dtype=torch.long), wpos=P, wmask=M)
        if wtags:
            Tg = torch.full((len(b), Wm), nd.IGNORE, dtype=torch.long)
            for i, x in enumerate(wtags):
                Tg[i, :len(x)] = torch.tensor(x, dtype=torch.long)
            out["wtags"] = Tg
            if head == "span":
                S = torch.zeros(len(b), Wm, maxw, dtype=torch.long)
                for i, ((d, s, L), (w0, w1)) in enumerate(zip(b, ranges)):
                    for gi, gj, l in docs[d]["wspans"]:
                        if gj < w0 or gi >= w1:
                            continue
                        if gi >= w0 and gj < w1:                    # entity inside the window
                            if gj - gi < maxw: S[i, gi - w0, gj - gi] = l
                        elif gi >= w0:                              # runs past the window end: unknown
                            S[i, gi - w0, :] = nd.IGNORE
                        else:                                       # started before the window: unknown
                            for a in range(max(w0, gj - maxw + 1), gj + 1):
                                S[i, a - w0, gj - a] = nd.IGNORE
                out["stgt"] = S
        return {k: v.to(device) for k, v in out.items()}, ranges


def length_batches(win, bs, shuffle, rng=None):
    by_len = {}
    for w in win:
        by_len.setdefault(w[2], []).append(w)
    out = []
    for _, ws in sorted(by_len.items()):
        if shuffle: rng.shuffle(ws)
        out += [ws[i:i + bs] for i in range(0, len(ws), bs)]
    if shuffle: rng.shuffle(out)
    return out


# ======================================================================================================
# 5. Prediction: windows -> document-level scores -> spans
# ======================================================================================================
@torch.no_grad()
def predict_scores(net, docs, wd, stride, bs, device):
    """returns per document: linear -> averaged tag probabilities (n_words,K); crf -> averaged emissions;
    span -> {(i,j): averaged class probabilities}. Overlapping windows are combined with a tent weight
    (a word in the middle of a window has more context than one at its edge)."""
    net.eval()
    model = net.student if isinstance(net, KDWrapper) else net
    head = model.head_type
    out = {}
    for b in length_batches(wd.windows(docs, stride), bs, shuffle=False):
        batch, ranges = wd.batch(docs, b, device)
        res = net(batch["ids"], batch["wpos"], batch["wmask"]).float().cpu()
        for k, ((d, s, L), (w0, w1)) in enumerate(zip(b, ranges)):
            n = w1 - w0
            if n == 0:
                continue
            r = docs[d]
            pos = torch.tensor([r["wtok"][w] - s for w in range(w0, w1)], dtype=torch.float)
            wt = 1 + torch.minimum(pos, L - 1 - pos)
            if head == "span":
                cand = out.setdefault(d, {})
                pr = res[k, :n].softmax(-1)                                 # (n, maxw, C)
                keep = (pr[..., 1:].max(-1).values > 0.2).nonzero().tolist()     # plausible entities only
                for i, kk in keep:
                    j = i + kk
                    if j >= n: continue
                    key = (w0 + i, w0 + j); ww = float(min(wt[i], wt[j]))
                    acc = cand.get(key)
                    cand[key] = [pr[i, kk] * ww, ww] if acc is None else [acc[0] + pr[i, kk] * ww, acc[1] + ww]
            else:
                if d not in out:
                    nw = len(r["wtok"]); out[d] = [torch.zeros(nw, res.size(-1)), torch.zeros(nw, 1)]
                v = res[k, :n].softmax(-1) if head == "linear" else res[k, :n]
                out[d][0][w0:w1] += v * wt[:, None]; out[d][1][w0:w1] += wt[:, None]
    final = {}
    for d in docs:
        if head == "span":
            final[d] = {key: a / w for key, (a, w) in out.get(d, {}).items()}
        else:
            v = out.get(d)
            final[d] = (v[0] / v[1].clamp(min=1e-9)) if v is not None else torch.zeros(0, len(model.tags))
    return final


def decode(scores, docs, model, texts, strategy="viterbi", propagate=False):
    """document scores -> {doc: [(s, e, label)]}.
    linear head: "greedy" (argmax per word + lenient BIO repair) or "viterbi" (best BIO-valid sequence
    under the word log-probabilities); crf head: Viterbi with the learned transitions; span head: greedy
    non-overlapping spans by probability. propagate: document-level label consistency (see ner_data)."""
    head, tags = model.head_type, model.tags
    id2tag = dict(enumerate(tags))
    pred = {}
    if head != "span":
        if head == "crf":
            trans, start, end = [x.detach().float().cpu() for x in model.crf.scores()]
        else:
            ok, start_ok = nd.allowed_transitions(tags)
            trans = (~torch.tensor(ok)).float() * NEG; start = (~torch.tensor(start_ok)).float() * NEG
            end = torch.zeros(len(tags))
    for d, sc in scores.items():
        r = docs[d]
        if head == "span":
            cands = {k: (float(v[1:].max()), int(v.argmax())) for k, v in sc.items()}
            spans = nd.spans_from_candidates(cands, r["wcs"], r["wce"], model.id2label)
        elif head == "linear" and strategy == "greedy":
            spans = nd.spans_from_tags(sc.argmax(-1).tolist(), r["wcs"], r["wce"], id2tag)
        else:
            em = sc.clamp(min=1e-9).log() if head == "linear" else sc
            spans = nd.spans_from_tags(viterbi(em, trans, start, end), r["wcs"], r["wce"], id2tag)
        pred[d] = nd.propagate(spans, texts[d]) if propagate else spans
    return pred


def decoding_options(head):
    base = ["greedy", "viterbi"] if head == "linear" else ["viterbi"] if head == "crf" else ["greedy"]
    return [(s, p) for s in base for p in (False, True)]


# ======================================================================================================
# 6. Training
# ======================================================================================================
def train(net, train_docs, val_docs, val_texts, val_gold, labels, wd, cfg, device, log=print):
    """AdamW (backbone/adaptation LR + head LR), linear warm-up and decay, fp16, gradient accumulation,
    DataParallel over all GPUs; evaluates strict macro-F1 on the validation documents after every epoch
    and keeps the best weights (early stopping with patience)."""
    rng = random.Random(cfg["seed"])
    model = net.student if isinstance(net, KDWrapper) else net
    gpus = torch.cuda.device_count()
    para = nn.DataParallel(net) if gpus > 1 else net
    is_head = lambda n: n.startswith(("head.", "crf.", "student.head.", "student.crf."))
    named = [(n, p) for n, p in net.named_parameters() if p.requires_grad]
    groups = [dict(params=[p for n, p in named if not is_head(n)], lr=cfg["lr"]),
              dict(params=[p for n, p in named if is_head(n)], lr=cfg["lr_head"])]
    groups = [g for g in groups if g["params"]]
    for g in groups: g["base_lr"] = g["lr"]
    params = [p for g in groups for p in g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    win = wd.windows(train_docs, cfg["train_stride"])
    acc = cfg.get("accum", 1)
    steps_per_epoch = math.ceil(len(win) / cfg["batch_size"] / acc)
    total = steps_per_epoch * cfg["epochs"]; warm = max(1, int(cfg["warmup_frac"] * total))
    log(f"train windows {len(win):,} of {wd.size} tokens (+{wd.H} header) | {steps_per_epoch} steps/epoch x "
        f"{cfg['epochs']} epochs | trainable {sum(p.numel() for p in params):,} | GPUs {max(gpus, 1)}")
    best, best_state, history, step, bad, t0 = -1.0, None, [], 0, 0, time.time()
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    maxw = getattr(model, "maxw", None)
    for ep in range(1, cfg["epochs"] + 1):
        net.train(); run, n = 0.0, 0
        bl = length_batches(win, cfg["batch_size"], shuffle=True, rng=rng)
        for bi, b in enumerate(bl):
            batch, _ = wd.batch(train_docs, b, device, model.head_type, maxw)
            loss = para(**batch).mean() / acc
            scaler.scale(loss).backward()
            run += loss.item() * acc; n += 1
            if (bi + 1) % acc and bi + 1 < len(bl):
                continue
            step += 1
            f = step / warm if step <= warm else max(0.0, (total - step) / max(1, total - warm))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * f
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, cfg["clip"])
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            if step % cfg["log_every"] == 0:
                log(f"  ep {ep} step {step}/{total} | loss {run / n:.4f} | {(time.time() - t0) / 60:.1f} min")
                run, n = 0.0, 0
        if cfg.get("select_last"):                    # refit on all documents: fixed epochs, no selection
            history.append(dict(epoch=ep, step=step, minutes=round((time.time() - t0) / 60, 1)))
            log(f"epoch {ep} done (refit on all documents: no validation-based selection)")
            continue
        sc = predict_scores(net, val_docs, wd, cfg["eval_stride"], cfg["eval_batch"], device)
        strategy = "viterbi" if model.head_type != "span" else "greedy"
        res = nd.strict_scores(decode(sc, val_docs, model, val_texts, strategy), val_gold, labels)
        history.append(dict(epoch=ep, step=step, macro_F1=res["macro_F1"], micro_F1=res["micro_F1"],
                            macro_P=res["macro_P"], macro_R=res["macro_R"], minutes=round((time.time() - t0) / 60, 1)))
        note = ""
        if res["macro_F1"] > best + 1e-4:
            best, bad, note = res["macro_F1"], 0, "  <- best"
            best_state = {k: v.detach().to("cpu", copy=True) for k, v in net.state_dict().items()}
        else:
            bad += 1
        log(f"epoch {ep}: val strict macro-F1 {res['macro_F1']:.4f} | micro {res['micro_F1']:.4f} | "
            f"P {res['macro_P']:.3f} R {res['macro_R']:.3f}{note}")
        if cfg.get("patience") and bad >= cfg["patience"]:
            log(f"early stopping: no improvement for {bad} epochs"); break
        if (time.time() - t0) / 3600 > cfg["max_train_hours"]:
            log("time budget reached - stopping"); break
    if best_state is not None:
        net.load_state_dict(best_state)
    peak = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
    return dict(best_val_macro_F1=best, history=history, train_minutes=round((time.time() - t0) / 60, 1),
                peak_gpu_gb=round(peak, 2))


def trainable_state(net):
    """only the trained tensors (what a PEFT method has to store per task)"""
    names = {n for n, p in net.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in net.state_dict().items() if k in names}


@torch.no_grad()
def self_test(net, wd, docs, device, head, maxw=None):
    """one forward + backward on two real windows before the long run (catches shape/device errors early)"""
    b = wd.windows(docs, wd.size)[:2]
    if len(b) == 2 and b[0][2] != b[1][2]: b = b[:1]
    batch, _ = wd.batch(docs, b, device, head, maxw)
    net.train()
    with torch.enable_grad():
        loss = net(**batch).mean(); loss.backward()
    net.zero_grad(set_to_none=True)
    out = net(batch["ids"], batch["wpos"], batch["wmask"])
    return float(loss), tuple(out.shape)

In [ ]:
%%writefile p3src/rr.py
# Phase 3 (Rhetorical Role prediction) - hierarchical multi-task model on top of a Phase-1 backbone.
#   sentences -> (packed into windows, each token sees its neighbouring sentences) -> Phase-1 backbone (+ PEFT)
#   -> mean-pooled sentence vectors (+ relative position + domain) -> document BiLSTM
#   -> role head (13 labels) + label-shift head (does the role change from the previous sentence?)
# Techniques: full FT / LoRA / rsLoRA / LoRMA / IA3 / BitFit / ... and knowledge distillation (soft role targets).
import json
import math
import os
import random
import time
import zipfile
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

import ner_model as nm

# ======================================================================================================
# 1. Data
# ======================================================================================================
_ID = ("id", "doc_id", "document_id", "docid", "name")


def parse_docs(raw):
    """RR json in any common shape -> list of dict(id, text=[sentences], labels=[...] or None)"""
    if isinstance(raw, dict) and "data" in raw and isinstance(raw["data"], (list, dict)):
        raw = raw["data"]
    items = []
    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, dict):
                items.append(dict(v, id=v.get("id", k)))
            else:                                                   # id -> list of sentences
                items.append(dict(id=k, text=v))
    else:
        items = list(raw)
    docs = []
    for i, d in enumerate(items):
        did = next((d[k] for k in _ID if k in d), str(i))
        text = d.get("text") or d.get("sentences") or d.get("sents")
        if isinstance(text, str):
            text = [text]
        labels = d.get("labels") or d.get("label") or d.get("roles")
        docs.append(dict(id=str(did), text=[str(s) for s in text], labels=list(labels) if labels else None))
    return docs


def parse_labels(raw):
    if isinstance(raw, dict):
        if "labels" in raw:
            return list(raw["labels"])
        if all(isinstance(v, int) for v in raw.values()):
            return [k for k, _ in sorted(raw.items(), key=lambda kv: kv[1])]
        return list(raw)
    return list(raw)


def encode_doc(tok, sentences, max_sent_tokens):
    """-> list of token-id lists, one per sentence (never empty, at most max_sent_tokens)"""
    space = getattr(tok, "space_prefix", False)
    out = []
    for s in sentences:
        s = " ".join(s.split())
        ids = tok.encode((" " + s) if space else s) if s else []
        if not space:
            ids = tok.encode(" ") + ids                             # A1 tokenizer: explicit separating space
        out.append((ids or tok.encode(" ."))[:max_sent_tokens])
    return out


_TOK = None


def lex_tokens(s, ngrams=2):
    """lower-cased word unigrams (+ bigrams); numbers -> <num>"""
    global _TOK
    if _TOK is None:
        import re
        _TOK = re.compile(r"[a-z]+|\d+")
    w = ["<num>" if t.isdigit() else t for t in _TOK.findall(s.lower())]
    return w + ([a + "_" + b for a, b in zip(w, w[1:])] if ngrams >= 2 else [])


def lex_vocab(docs, max_features=30000, min_df=3, ngrams=2):
    """n-gram vocabulary + idf from the TRAINING sentences only"""
    df, n = Counter(), 0
    for d in docs:
        for s in d["text"]:
            df.update(set(lex_tokens(s, ngrams))); n += 1
    keep = [f for f, c in df.most_common() if c >= min_df][:max_features]
    return dict(index={f: i for i, f in enumerate(keep)}, idf=[math.log((1 + n) / (1 + df[f])) + 1 for f in keep],
                ngrams=ngrams, key=(max_features, min_df, ngrams, n))


def lex_encode(vocab, s):
    """sentence -> (feature ids, l2-normalised sublinear tf-idf weights)"""
    c = Counter(f for f in lex_tokens(s, vocab["ngrams"]) if f in vocab["index"])
    ids = [vocab["index"][f] for f in c]
    w = [(1 + math.log(c[f])) * vocab["idf"][vocab["index"][f]] for f in c]
    norm = math.sqrt(sum(x * x for x in w)) or 1.0
    return ids, [x / norm for x in w]


def lex_batch(it, s0, s1, device):
    """bag of n-grams of sentences s0..s1-1 of a prepared document -> EmbeddingBag inputs"""
    ids, offs, ws = [], [], []
    for i in range(s0, s1):
        a, b = it["lex"][i]; offs.append(len(ids)); ids += a; ws += b
    return (torch.tensor(ids, dtype=torch.long, device=device), torch.tensor(offs, dtype=torch.long, device=device),
            torch.tensor(ws, dtype=torch.float, device=device))


def pack_windows(sent_ids, size):
    """greedy: consecutive whole sentences into windows of <= size tokens -> list of (first, last+1) sentence ranges"""
    wins, start, n = [], 0, 0
    for i, ids in enumerate(sent_ids):
        if n + len(ids) > size and i > start:
            wins.append((start, i)); start, n = i, 0
        n += len(ids)
    wins.append((start, len(sent_ids)))
    return wins


def window_batch(sent_ids, wins, device, pad_id=0):
    """windows -> ids (B,T), seg (B,T) slot of the sentence each token belongs to (-1 = padding), smask (B,S)"""
    B = len(wins)
    T = max(sum(len(sent_ids[i]) for i in range(a, b)) for a, b in wins)
    S = max(b - a for a, b in wins)
    ids = torch.full((B, T), pad_id, dtype=torch.long)
    seg = torch.full((B, T), -1, dtype=torch.long)
    smask = torch.zeros(B, S, dtype=torch.bool)
    for r, (a, b) in enumerate(wins):
        p = 0
        for k, i in enumerate(range(a, b)):
            L = len(sent_ids[i])
            ids[r, p:p + L] = torch.tensor(sent_ids[i]); seg[r, p:p + L] = k; p += L
            smask[r, k] = True
    return ids.to(device), seg.to(device), smask.to(device)


# ======================================================================================================
# 2. Extra parameter-efficient methods (the others come from ner_model.apply_method)
# ======================================================================================================
class RSLoRALinear(nm.LoRALinear):
    """rank-stabilised LoRA (Kalajdzievski 2023): scale alpha / sqrt(r) instead of alpha / r"""

    def __init__(self, base, r=16, alpha=16, dropout=0.05):
        super().__init__(base, r, alpha, dropout)
        self.scale = alpha / math.sqrt(r)


class OutScale(nn.Module):
    """IA3 (Liu et al. 2022): multiply a module's output by a learned vector (initialised to 1)"""

    def __init__(self, module, dim):
        super().__init__()
        self.module = module
        self.l = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return self.module(x) * self.l


def apply_method(model, cfg):
    bb, method = model.backbone, cfg["method"]
    if method == "rslora":
        for p in bb.parameters(): p.requires_grad = False
        r, a = cfg.get("rank", 16), cfg.get("alpha", 16)
        n = sum(nm._replace_linears(blk, lambda lin: RSLoRALinear(lin, r, a)) for blk in bb.blocks)
        for mod in bb.modules():
            if isinstance(mod, RSLoRALinear):
                mod.A.requires_grad = True; mod.B.requires_grad = True
        return f"rslora (rank {r}, alpha {a}, scale alpha/sqrt(r), {n} linear layers)"
    if method == "ia3":
        for p in bb.parameters(): p.requires_grad = False
        n = 0
        for blk in bb.blocks:
            for parent in list(blk.modules()):
                for name, child in list(parent.named_children()):
                    if name in ("k_proj", "v_proj", "kv_proj", "k_up", "v_up") and isinstance(child, nn.Linear):
                        setattr(parent, name, OutScale(child, child.out_features)); n += 1
                    elif type(child).__name__ == "SwiGLU":                     # gated FFN: scale the gate branch
                        child.w3 = OutScale(child.w3, child.w3.out_features); n += 1
                    elif name == "net" and isinstance(child, nn.Sequential) and isinstance(child[0], nn.Linear):
                        child[1] = OutScale(child[1], child[0].out_features); n += 1   # GELU FFN: after activation
        for mod in bb.modules():
            if isinstance(mod, OutScale):
                mod.l.requires_grad = True
        return f"ia3 ({n} scaling vectors on keys, values and FFN activations)"
    return nm.apply_method(model, cfg)


# ======================================================================================================
# 3. Model
# ======================================================================================================
class SentenceEncoder(nn.Module):
    """backbone over packed windows -> mean of each sentence's token states. forward(ids, seg, smask) -> (B,S,d)"""

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, ids, seg, smask):
        with nm.autocast():
            h = self.backbone(ids)
        S = smask.size(1)
        m = (seg[:, :, None] == torch.arange(S, device=ids.device)[None, None]).to(h.dtype)   # (B,T,S)
        pooled = torch.einsum("bts,btd->bsd", m, h) / m.sum(1).clamp(min=1)[..., None]
        return pooled.float()


class CRF(nn.Module):
    """linear-chain CRF over a document's sentences (Malik et al. 2022: BiLSTM -> BiLSTM-CRF gave +0.04..0.07).
    trans[i, j] = score of label j following label i."""

    def __init__(self, n_labels):
        super().__init__()
        self.trans = nn.Parameter(torch.zeros(n_labels, n_labels))
        self.start = nn.Parameter(torch.zeros(n_labels))
        self.end = nn.Parameter(torch.zeros(n_labels))

    def nll(self, e, y):
        """e (N,L) emission scores, y (N,) gold -> negative log-likelihood per sentence"""
        N = e.size(0)
        gold = self.start[y[0]] + e[torch.arange(N, device=e.device), y].sum() + self.end[y[-1]]
        if N > 1:
            gold = gold + self.trans[y[:-1], y[1:]].sum()
        a = self.start + e[0]
        for i in range(1, N):
            a = torch.logsumexp(a[:, None] + self.trans, 0) + e[i]
        Z = torch.logsumexp(a + self.end, 0)
        return (Z - gold) / N

    def export(self):
        return dict(trans=self.trans.detach().float().cpu().clone(), start=self.start.detach().float().cpu().clone(),
                    end=self.end.detach().float().cpu().clone())


class RRModel(nn.Module):
    def __init__(self, backbone, n_labels, hidden=256, dropout=0.3, use_bilstm=True, use_lsp=True,
                 class_weights=None, lsp_weight=0.5, n_domains=2, use_pos=True, use_crf=False, crf_weight=1.0,
                 lex_size=0, lex_dim=0):
        super().__init__()
        self.crf = CRF(n_labels) if use_crf else None
        self.crf_weight = crf_weight
        self.lex = None
        if lex_dim:                                    # lexical channel: tf-idf weighted bag of word n-grams -> lex_dim
            self.lex = nn.EmbeddingBag(lex_size, lex_dim, mode="sum")
            nn.init.normal_(self.lex.weight, std=0.1)
            self.lex_drop = nn.Dropout(dropout)
        self.use_pos = use_pos
        self.backbone = backbone                       # (also inside the encoder; same object)
        self.encoder = SentenceEncoder(backbone)
        self.use_bilstm, self.use_lsp, self.lsp_weight = use_bilstm, use_lsp, lsp_weight
        d = backbone.d
        self.proj = nn.Sequential(nn.Linear(d + 3 + n_domains + lex_dim, hidden * 2), nn.GELU(), nn.Dropout(dropout))
        if use_bilstm:
            self.lstm = nn.LSTM(hidden * 2, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden * 2, n_labels)
        self.shift_head = nn.Linear(hidden * 6, 2)
        self.register_buffer("cw", class_weights if class_weights is not None else torch.ones(n_labels))
        self.n_domains = n_domains

    def doc_logits(self, sv, domain, start=0, total=None, lex=None):
        """sv (N,d) vectors of consecutive sentences start..start+N-1 of a document with `total` sentences
        -> role logits (N,L), shift logits (N,2). Position = absolute relative position in the whole document.
        lex: (ids, offsets, weights) bag of n-grams of the same N sentences (only if the lexical channel is on)."""
        N = sv.size(0); total = total or N
        pos = (start + torch.arange(N, device=sv.device, dtype=torch.float)) / max(1, total - 1)
        feats = torch.stack([pos, (pos * math.pi).sin(), (pos * math.pi).cos()], 1) * float(self.use_pos)
        dom = F.one_hot(torch.full((N,), domain, device=sv.device), self.n_domains).float()
        parts = [sv, feats, dom]
        if self.lex is not None:
            parts.append(self.lex_drop(self.lex(lex[0], lex[1], per_sample_weights=lex[2])))
        x = self.proj(torch.cat(parts, 1))
        if self.use_bilstm:
            x, _ = self.lstm(x[None]); x = x[0]
        x = self.drop(x)
        prev = torch.cat([x[:1], x[:-1]], 0)
        shift = self.shift_head(torch.cat([x, prev, x - prev], 1))
        return self.head(x), shift

    def loss(self, role_logits, shift_logits, labels, soft=None, kd_T=2.0, kd_alpha=0.5):
        ce = F.cross_entropy(role_logits, labels, weight=self.cw)
        if soft is not None:                               # knowledge distillation from teacher probabilities
            kl = F.kl_div(F.log_softmax(role_logits / kd_T, -1), soft, reduction="batchmean") * kd_T ** 2
            ce = kd_alpha * ce + (1 - kd_alpha) * kl
        if self.use_lsp and labels.numel() > 1:
            shift_gold = (labels[1:] != labels[:-1]).long()
            ce = ce + self.lsp_weight * F.cross_entropy(shift_logits[1:], shift_gold)
        if self.crf is not None:
            ce = ce + self.crf_weight * self.crf.nll(role_logits, labels)
        return ce


# ======================================================================================================
# 3b. Decoding: Viterbi with the CRF transitions + guard against stray very-rare labels
# ======================================================================================================
# PrecedentOverruled: 3 training sentences, 0 dev -> never predicted (a wrong prediction adds a 0-F1 class to the
# macro average). Dissent: in the training data it is a long contiguous block (a dissenting note), so fewer than
# 5 Dissent sentences in a document are treated as noise and re-decoded without that label.
GUARD = dict(forbid=("PrecedentOverruled",), min_block={"Dissent": 5})


def viterbi(lp, tr, lam=1.0):
    """lp (N,L) log-probabilities, tr dict(trans, start, end) -> best label path (list of ids)"""
    T, st, en = tr["trans"] * lam, tr["start"] * lam, tr["end"] * lam
    score = st + lp[0]; back = []
    for i in range(1, lp.size(0)):
        cand = score[:, None] + T
        best, idx = cand.max(0)
        score = best + lp[i]; back.append(idx)
    y = int((score + en).argmax())
    path = [y]
    for idx in reversed(back):
        y = int(idx[y]); path.append(y)
    return path[::-1]


def decode_doc(p, labels, tr=None, lam=1.0, guard=None):
    """probabilities (N,L) of one document -> label ids (Viterbi if transitions given, else argmax)"""
    lp = p.float().clamp_min(1e-9).log()
    if guard:
        for l in guard.get("forbid", ()):
            if l in labels: lp[:, labels.index(l)] = -1e4
    dec = (lambda x: viterbi(x, tr, lam)) if tr is not None and lam > 0 else (lambda x: x.argmax(-1).tolist())
    path = dec(lp)
    if guard:
        for l, k in guard.get("min_block", {}).items():
            if l in labels:
                i = labels.index(l); c = sum(1 for y in path if y == i)
                if 0 < c < k:
                    lp[:, i] = -1e4; path = dec(lp)
    return path


def combine(prob_sets, weights=None):
    """weighted geometric mean of several {doc_id: (N,L) probs} -> normalised probs"""
    keys = list(prob_sets[0])
    w = weights or [1.0 / len(prob_sets)] * len(prob_sets)
    return {k: sum(wi * ps[k].float().clamp_min(1e-9).log() for wi, ps in zip(w, prob_sets)).softmax(-1) for k in keys}


def mean_trans(trs):
    trs = [t for t in trs if t is not None]
    if not trs:
        return None
    return {k: sum(t[k] for t in trs) / len(trs) for k in ("trans", "start", "end")}


# ======================================================================================================
# 4. Metrics
# ======================================================================================================
def macro_scores(gold, pred, labels):
    """sklearn-style macro P/R/F1 over the labels present in gold or prediction (flattened sentences)"""
    tp, fp, fn = Counter(), Counter(), Counter()
    for g, p in zip(gold, pred):
        if g == p: tp[g] += 1
        else: fp[p] += 1; fn[g] += 1
    rows = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        rows[l] = dict(P=P, R=R, F1=2 * P * R / (P + R) if P + R else 0.0, support=tp[l] + fn[l])
    used = [l for l in labels if rows[l]["support"] or fp[l]]
    m = lambda k: sum(rows[l][k] for l in used) / max(1, len(used))
    acc = sum(tp.values()) / max(1, len(gold))
    return dict(macro_F1=m("F1"), macro_P=m("P"), macro_R=m("R"), accuracy=acc, per_label=rows)


def evaluate(probs, docs, labels, tr=None, lam=1.0, guard=None):
    """probs {doc_id: (n, L)} -> scores overall and per domain + predictions (Viterbi if transitions tr given)"""
    out, pred = {}, {}
    ids = {d["id"]: decode_doc(probs[d["id"]], labels, tr, lam, guard) for d in docs}
    for dom in sorted({d["domain"] for d in docs}) + ["ALL"]:
        g, p = [], []
        for d in docs:
            if dom != "ALL" and d["domain"] != dom:
                continue
            pl = [labels[i] for i in ids[d["id"]]]
            pred[d["id"]] = pl
            g += d["labels"]; p += pl
        out[dom] = macro_scores(g, p, labels)
    return out, pred


def confusion(docs, pred, top=10):
    c = Counter()
    for d in docs:
        for g, p in zip(d["labels"], pred[d["id"]]):
            if g != p: c[(g, p)] += 1
    return c.most_common(top)


# ======================================================================================================
# 5. Training / prediction
# ======================================================================================================
def _lex(model, it, s0, s1, device):
    return lex_batch(it, s0, s1, device) if getattr(model, "lex", None) is not None else None


LEX = None                                              # n-gram vocabulary of the lexical channel (training sentences)


def ensure_lex(D, cfg, preps, log=print):
    """build the n-gram vocabulary from the training documents once and attach bags to every prepared document"""
    global LEX
    key = (cfg.get("lex_features", 30000), cfg.get("lex_min_df", 3), cfg.get("lex_ngrams", 2))
    n_train = sum(len(d["text"]) for d in D["train"])           # smoke-check data and real data get different vocabularies
    if LEX is None or LEX["key"] != key + (n_train,):
        LEX = lex_vocab(D["train"], *key)
        log(f"lexical channel: {len(LEX['index']):,} word uni/bigram features from the training sentences")
    for prep in preps:
        for it in prep.items:
            if it.get("lex_key") != LEX["key"]:
                it["lex"] = [lex_encode(LEX, s) for s in it["doc"]["text"]]; it["lex_key"] = LEX["key"]


def label_prior(D, smooth=1.0):
    c = Counter(l for d in D["train"] for l in d["labels"])
    p = torch.tensor([c[l] + smooth for l in D["labels"]], dtype=torch.float)
    return p / p.sum()


def prior_adjust(probs, prior, tau):
    """logit adjustment for macro-F1: p(y|x) / prior(y)^tau (tau = 0: unchanged)"""
    if not tau:
        return probs
    lp = prior.log() * tau
    return {k: (v.float().clamp_min(1e-9).log() - lp).softmax(-1) for k, v in probs.items()}


def tune_decode(dev, D, tr, prior, taus=(0.0, 0.25, 0.5, 0.75), lams=(0.0, 0.5, 1.0, 1.5)):
    """grid over prior correction tau and CRF weight lambda on dev -> best (tau, lam, F1), full table"""
    best, table = None, []
    for tau in taus:
        dv = prior_adjust(dev, prior, tau)
        for lam in (lams if tr is not None else (0.0,)):
            f = evaluate(dv, D["dev"], D["labels"], tr, lam, GUARD)[0]["ALL"]["macro_F1"]
            table.append((tau, lam, round(f, 4)))
            if best is None or f > best[2] + 1e-9:
                best = (tau, lam, f)
    return best, table


class Prepared:
    """tokenised documents for one tokenizer: sentence ids + windows"""

    def __init__(self, docs, tok, window, max_sent_tokens, label2id, sentence_only=False):
        self.items = []
        for d in docs:
            sids = encode_doc(tok, d["text"], max_sent_tokens)
            y = torch.tensor([label2id[l] for l in d["labels"]]) if d.get("labels") else None
            wins = [(i, i + 1) for i in range(len(sids))] if sentence_only else pack_windows(sids, window)
            self.items.append(dict(doc=d, sids=sids, wins=wins, y=y))


@torch.no_grad()
def predict(model, enc, prep, device, eval_batch=16):
    """{doc_id: (n_sentences, L) probabilities}"""
    model.eval()
    out = {}
    for it in prep.items:
        vecs = []
        for i in range(0, len(it["wins"]), eval_batch):
            ids, seg, smask = window_batch(it["sids"], it["wins"][i:i + eval_batch], device)
            pooled = enc(ids, seg, smask)
            vecs.append(pooled[smask])
        sv = torch.cat(vecs, 0)
        logits, _ = model.doc_logits(sv, it["doc"]["dom_id"], lex=_lex(model, it, 0, len(it["sids"]), device))
        out[it["doc"]["id"]] = logits.float().softmax(-1).cpu()
        assert out[it["doc"]["id"]].size(0) == len(it["doc"]["text"])
    return out


def train(model, train_prep, dev_prep, dev_docs, labels, cfg, device, log=print, soft_targets=None, test_prep=None):
    rng = random.Random(cfg["seed"])
    gpus = torch.cuda.device_count()
    enc = nn.DataParallel(model.encoder) if gpus > 1 else model.encoder
    is_bb = lambda n: n.startswith(("backbone.", "encoder."))
    named = {}
    for n, p in model.named_parameters():
        if p.requires_grad and id(p) not in {id(q) for q in named.values()}:
            named[n] = p
    groups = [dict(params=[p for n, p in named.items() if is_bb(n)], lr=cfg["lr"]),
              dict(params=[p for n, p in named.items() if not is_bb(n)], lr=cfg["lr_head"])]
    groups = [g for g in groups if g["params"]]
    for g in groups: g["base_lr"] = g["lr"]
    params = [p for g in groups for p in g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    acc = cfg["accum"]
    steps_per_epoch = math.ceil(len(train_prep.items) / acc)
    total = steps_per_epoch * cfg["epochs"]; warm = max(1, int(cfg["warmup_frac"] * total))
    log(f"train documents {len(train_prep.items)} | {steps_per_epoch} steps/epoch x {cfg['epochs']} epochs | "
        f"trainable {sum(p.numel() for p in params):,} | GPUs {max(gpus, 1)}")
    best, best_state, history, step, bad, t0 = -1.0, None, [], 0, 0, time.time()
    snaps, K = [], cfg.get("snapshot_k", 1)              # top-K epochs by dev: their dev/test probabilities
    guard = GUARD if cfg.get("guard") else None
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

    def step_doc(it, wins):
        s0, s1 = wins[0][0], wins[-1][1]
        ids, seg, smask = window_batch(it["sids"], wins, device)
        sv = enc(ids, seg, smask)[smask]
        role, shift = model.doc_logits(sv, it["doc"]["dom_id"], s0, len(it["sids"]), _lex(model, it, s0, s1, device))
        y = it["y"][s0:s1].to(device)
        soft = soft_targets[it["doc"]["id"]][s0:s1].to(device) if soft_targets is not None else None
        loss = model.loss(role.float(), shift.float(), y, soft, cfg.get("kd_T", 2.0), cfg.get("kd_alpha", 0.5)) / acc
        scaler.scale(loss).backward()
        return loss.item() * acc

    for ep in range(1, cfg["epochs"] + 1):
        model.train(); order = list(range(len(train_prep.items))); rng.shuffle(order); run = n = 0
        for k, idx in enumerate(order):
            it = train_prep.items[idx]
            W = len(it["wins"]); mw = cfg["max_windows"]
            a = rng.randrange(0, max(1, W - mw + 1))
            wins = it["wins"][a:a + mw]
            try:
                run += step_doc(it, wins)
            except torch.cuda.OutOfMemoryError:                      # very long document: retry with half of it
                opt.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                log(f"  (out of memory on {len(wins)} windows of {it['doc']['id']} -> retry with {max(1, len(wins) // 2)})")
                run += step_doc(it, wins[:max(1, len(wins) // 2)])
            n += 1
            if (k + 1) % acc and k + 1 < len(order):
                continue
            step += 1
            f = step / warm if step <= warm else max(0.0, (total - step) / max(1, total - warm))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * f
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, cfg["clip"])
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        probs = predict(model, model.encoder, dev_prep, device, cfg["eval_batch"])
        tr = model.crf.export() if model.crf is not None else None
        sc, _ = evaluate(probs, dev_docs, labels, tr, 1.0, guard)
        line = " | ".join(f"{k} {v['macro_F1']:.4f}" for k, v in sc.items())
        history.append(dict(epoch=ep, loss=run / max(1, n), **{k: v["macro_F1"] for k, v in sc.items()},
                            minutes=round((time.time() - t0) / 60, 1)))
        f1 = sc["ALL"]["macro_F1"]
        if cfg.get("select_last"):                  # refit on train+dev: dev is in-sample, so keep the LAST epochs
            f1 = float(ep)
        if K > 1 and (len(snaps) < K or f1 > min(s["f1"] for s in snaps)):
            snaps.append(dict(epoch=ep, f1=f1, dev=probs, tr=tr,
                              test=predict(model, model.encoder, test_prep, device, cfg["eval_batch"])))
            snaps = sorted(snaps, key=lambda s: -s["f1"])[:K]
        note = ""
        if f1 > best + 1e-4:
            best, bad, note = f1, 0, "  <- best"
            best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
        else:
            bad += 1
        log(f"epoch {ep}: loss {run / max(1, n):.4f} | dev macro-F1 {line}{note} | {(time.time() - t0) / 60:.1f} min")
        if cfg.get("patience") and bad >= cfg["patience"]:
            log(f"early stopping after {bad} epochs without improvement"); break
    model.load_state_dict(best_state)
    peak = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
    snap = None
    if K > 1:
        snap = dict(epochs=[s["epoch"] for s in snaps], dev=combine([s["dev"] for s in snaps]),
                    test=combine([s["test"] for s in snaps]), tr=mean_trans([s["tr"] for s in snaps]))
        log(f"averaging the top-{len(snaps)} epochs {snap['epochs']} (dev macro-F1 {[round(s['f1'], 4) for s in snaps]})")
    return dict(best_dev_macro_F1=best, history=history, train_minutes=round((time.time() - t0) / 60, 1),
                peak_gpu_gb=round(peak, 2)), snap


# ======================================================================================================
# 6. One experiment
# ======================================================================================================
SCORES = {}                                             # run -> dict(dev=probs, test=probs)
_PREP = {}


def prepared(D, arch, cfg, label2id):
    size = min(cfg["window"], nm.ARCH_INFO[arch][1])
    so = bool(cfg.get("sentence_only"))
    key = (nm.ARCH_INFO[arch][0], size, cfg["max_sent_tokens"], so, tuple(d["id"] for d in D["train"]))
    if key not in _PREP:
        tok = nm.load_tokenizer(arch, D["root"])
        _PREP[key] = tuple(Prepared(D[k], tok, size, cfg["max_sent_tokens"], label2id, so) for k in ("train", "dev", "test"))
    return _PREP[key]


def build(cfg, D):
    labels = D["labels"]
    bb = nm.Backbone(cfg["arch"], D["root"], bidirectional=cfg["bidirectional"], keep_layers=cfg.get("keep_layers"),
                     init=cfg.get("init", "pretrained"))
    cw = None
    if cfg.get("class_weighting") == "sqrt_inv":
        c = Counter(l for d in D["train"] for l in d["labels"]); tot = sum(c.values())
        cw = torch.tensor([math.sqrt(tot / max(1, c[l])) for l in labels]); cw = cw / cw.mean()
    model = RRModel(bb, len(labels), hidden=cfg["hidden"], dropout=cfg["dropout"], use_bilstm=cfg["bilstm"],
                    use_lsp=cfg["lsp"], class_weights=cw, lsp_weight=cfg["lsp_weight"], use_pos=cfg.get("use_pos", True),
                    use_crf=cfg.get("crf", False), crf_weight=cfg.get("crf_weight", 1.0),
                    lex_size=len(LEX["index"]) if cfg.get("lex_dim") else 0, lex_dim=cfg.get("lex_dim", 0))
    desc = apply_method(model, cfg)
    return model, desc


def run_experiment(cfg, D, device, out_root, log=print, soft_targets=None, keep_model=False):
    t_start = time.time()
    run_dir = os.path.join(out_root, cfg["run"]); os.makedirs(run_dir, exist_ok=True)
    labels = D["labels"]; label2id = {l: i for i, l in enumerate(labels)}
    tr, dv, te = prepared(D, cfg["arch"], cfg, label2id)
    if cfg.get("lex_dim"):
        ensure_lex(D, cfg, (tr, dv, te), log)
    torch.manual_seed(cfg["seed"]); random.seed(cfg["seed"])
    model, desc = build(cfg, D)
    model.to(device)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"\n=== {cfg['run']} | {cfg['arch']} | {desc} | BiLSTM {cfg['bilstm']} | LSP {cfg['lsp']} | "
        f"CRF {cfg.get('crf', False)} | lexical {cfg.get('lex_dim', 0)} | params {total:,}, trainable {trainable:,} ({trainable / total:.2%})"
        + (" | KD" if soft_targets else ""))
    res, snap = train(model, tr, dv, D["dev"], labels, cfg, device, log, soft_targets, te)
    t0 = time.time()
    dev_probs = predict(model, model.encoder, dv, device, cfg["eval_batch"]); infer = time.time() - t0
    test_probs = predict(model, model.encoder, te, device, cfg["eval_batch"])
    trans = model.crf.export() if model.crf is not None else None
    if snap is not None:                                    # average of the top-K epochs replaces the single best
        dev_probs, test_probs, trans = snap["dev"], snap["test"], snap["tr"]
        res["snapshot_epochs"] = snap["epochs"]
    guard = GUARD if cfg.get("guard") else None
    SCORES[cfg["run"]] = dict(dev=dev_probs, test=test_probs, tr=trans)
    if cfg.get("keep_train"):                               # in-sample probabilities (teacher targets for distillation)
        SCORES[cfg["run"]]["train"] = predict(model, model.encoder, tr, device, cfg["eval_batch"])
    sc, dpred = evaluate(dev_probs, D["dev"], labels, trans, 1.0, guard)
    for dom, s in sc.items():
        log(f"DEV {dom:>4}: macro-F1 {s['macro_F1']:.4f} | P {s['macro_P']:.4f} | R {s['macro_R']:.4f} | acc {s['accuracy']:.4f}")
    log(f"{'label':<24}{'P':>7}{'R':>7}{'F1':>7}{'support':>9}")
    for l in labels:
        r = sc["ALL"]["per_label"][l]; log(f"{l:<24}{r['P']:7.3f}{r['R']:7.3f}{r['F1']:7.3f}{r['support']:9d}")
    conf = confusion(D["dev"], dpred)
    log(f"most confused (gold -> predicted): {conf}")
    zp = write_submission({cfg["run"]: test_probs}, D, os.path.join(out_root, cfg["run"]), cfg["run"], log,
                          tr=trans, guard=guard)
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()
                if k in {n for n, p in model.named_parameters() if p.requires_grad}},
               os.path.join(run_dir, "trainable_weights.pt"))
    rec = dict(run=cfg["run"], arch=cfg["arch"], method=desc, cfg=cfg, params_total=total, params_trainable=trainable,
               dev={k: dict(macro_F1=v["macro_F1"], macro_P=v["macro_P"], macro_R=v["macro_R"], accuracy=v["accuracy"])
                    for k, v in sc.items()},
               per_label={l: sc["ALL"]["per_label"][l] for l in labels}, confusion=[[a, b, n] for (a, b), n in conf],
               dev_inference_sec=round(infer, 1), kd=soft_targets is not None, zip=os.path.basename(zp),
               minutes_total=round((time.time() - t_start) / 60, 1), **res)
    json.dump(rec, open(os.path.join(run_dir, "record.json"), "w"), indent=1)
    if keep_model:
        return rec, model
    del model; torch.cuda.empty_cache()
    return rec, None


def write_submission(prob_sets, D, out_dir, name, log=print, weights=None, tr=None, lam=1.0, guard=None):
    """combine the test probabilities of one or more runs -> prediction.json {doc_id: [labels]} -> zip.
    Without transitions/guard: arithmetic mean + argmax (v1 behaviour). With them: geometric mean + Viterbi."""
    labels = D["labels"]
    runs = list(prob_sets)
    sub = {}
    geo = combine([prob_sets[r] for r in runs], weights) if (tr is not None or guard) else None
    for d in D["test"]:
        if geo is not None:
            pl = [labels[i] for i in decode_doc(geo[d["id"]], labels, tr, lam, guard)]
        else:
            p = sum(prob_sets[r][d["id"]] * (weights[i] if weights else 1.0) for i, r in enumerate(runs))
            pl = [labels[i] for i in p.argmax(-1).tolist()]
        assert len(pl) == len(d["text"]), d["id"]
        sub[d["id"]] = pl
    assert len(sub) == len(D["test"]), "duplicate test document ids"
    os.makedirs(out_dir, exist_ok=True)
    pj = os.path.join(out_dir, "prediction.json"); json.dump(sub, open(pj, "w"), ensure_ascii=False)
    zp = os.path.join(out_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        chk = json.loads(z.read("prediction.json"))
        assert z.namelist() == ["prediction.json"] and all(len(chk[d["id"]]) == len(d["text"]) for d in D["test"])
        assert all(l in labels for v in chk.values() for l in v)
    log(f"{name}: {len(sub)} test documents, {sum(map(len, sub.values())):,} sentences -> {zp}")
    for dom in ("CL", "IT"):
        c = Counter(l for d in D["test"] if d.get("domain") == dom for l in sub[d["id"]])
        log(f"   predicted test labels {dom}: {dict(c.most_common())}")
    return zp


def save_probs(records, cfgs, path):
    """dev/test (+train, transitions) probabilities of finished runs -> one file, input of later notebooks"""
    out = {}
    recs = {r["run"]: r for r in records}
    for c in cfgs:
        r = c["run"]
        if r not in recs or r not in SCORES:
            continue
        s = SCORES[r]
        out[r] = dict(arch=c["arch"], method=c["method"], crf=bool(c.get("crf")),
                      dev_F1=recs[r]["dev"]["ALL"]["macro_F1"], params=recs[r]["params_total"],
                      dev_inference_sec=recs[r]["dev_inference_sec"],
                      dev={k: v.float().cpu() for k, v in s["dev"].items()},
                      test={k: v.float().cpu() for k, v in s["test"].items()}, tr=s.get("tr"),
                      train={k: v.float().cpu() for k, v in s["train"].items()} if "train" in s else None)
    torch.save(out, path)
    return out


def load_probs(pattern="/kaggle/input/**/p3_probs_*.pt", log=print):
    """all saved probability files (v2 and v3 formats) -> {run: dict}"""
    import glob
    out = {}
    for f in sorted(glob.glob(pattern, recursive=True)):
        for n, v in torch.load(f, map_location="cpu", weights_only=False).items():
            if n not in out:
                v.setdefault("tr", None); v.setdefault("train", None); v.setdefault("params", None); out[n] = v
        log(f"loaded {f}")
    return out


def ensemble_dev(runs, D):
    probs = {d["id"]: sum(SCORES[r]["dev"][d["id"]] for r in runs) / len(runs) for d in D["dev"]}
    return evaluate(probs, D["dev"], D["labels"])[0]


def teacher_targets(model, D, cfg, device, T=2.0):
    """soft role targets of a trained teacher for every TRAINING document (for distillation)"""
    labels = D["labels"]; label2id = {l: i for i, l in enumerate(labels)}
    tr, _, _ = prepared(D, cfg["arch"], cfg, label2id)
    model.eval(); out = {}
    with torch.no_grad():
        for it in tr.items:
            vecs = []
            for i in range(0, len(it["wins"]), cfg["eval_batch"]):
                ids, seg, smask = window_batch(it["sids"], it["wins"][i:i + cfg["eval_batch"]], device)
                vecs.append(model.encoder(ids, seg, smask)[smask])
            logits, _ = model.doc_logits(torch.cat(vecs, 0), it["doc"]["dom_id"],
                                         lex=_lex(model, it, 0, len(it["sids"]), device))
            out[it["doc"]["id"]] = (logits.float() / T).softmax(-1).cpu()
    return out


def summary(records, log=print):
    log(f"\n{'run':<36}{'method':<40}{'trainable':>12}{'dev ALL':>9}{'CL':>8}{'IT':>8}{'min':>7}")
    for r in records:
        d = r["dev"]
        log(f"{r['run']:<36}{r['method'][:39]:<40}{r['params_trainable']:>12,}{d['ALL']['macro_F1']:>9.4f}"
            f"{d.get('CL', {}).get('macro_F1', 0):>8.4f}{d.get('IT', {}).get('macro_F1', 0):>8.4f}{r['minutes_total']:>7.1f}")

In [ ]:
import importlib
sys.path.insert(0, os.path.abspath("p3src"))
import ner_data as nd, ner_model as nm, rr
for _m in (nd, nm, rr): importlib.reload(_m)
print("Phase-3 code imported")

In [ ]:
# 4 · Load Phase_3_RR (CL + IT): train / dev / test + label vocabulary, with checks
from collections import Counter
from huggingface_hub import HfApi, hf_hub_download
REPO = "Exploration-Lab/CS-781-Capstone"
files = [f for f in HfApi(token=HF_TOKEN).list_repo_files(REPO, repo_type="dataset") if f.startswith("Phase_3")]
print("\n".join(files))
get = lambda n: hf_hub_download(REPO, next(f for f in files if f.endswith(n)), repo_type="dataset", token=HF_TOKEN)
load = lambda n: json.load(open(get(n), encoding="utf-8"))
LABELS = rr.parse_labels(load("label_vocab.json"))
print("labels:", LABELS)
D = dict(labels=LABELS, root=ROOT, train=[], dev=[], test=[])
for dom_id, dom in enumerate(("CL", "IT")):
    for split in ("train", "dev", "test"):
        docs = rr.parse_docs(load(f"{dom}_{split}.json"))
        for d in docs:
            d["domain"], d["dom_id"] = dom, dom_id
            if split != "test":
                assert d["labels"] and len(d["labels"]) == len(d["text"]), (dom, split, d["id"])
                assert set(d["labels"]) <= set(LABELS), set(d["labels"]) - set(LABELS)
        D[split] += docs
        n_s = sum(len(d["text"]) for d in docs)
        print(f"{dom} {split:<5}: {len(docs):4d} documents, {n_s:6d} sentences ({n_s / max(1, len(docs)):.0f} per document)")
assert len({d["id"] for d in D["test"]}) == len(D["test"]), "test document ids collide across CL and IT"
for split in ("train", "dev"):
    for dom in ("CL", "IT"):
        c = Counter(l for d in D[split] if d["domain"] == dom for l in d["labels"])
        print(f"{split} {dom} label counts:", dict(c.most_common()))
ex = D["train"][0]; print("\nexample:", ex["id"], list(zip(ex["text"][:3], ex["labels"][:3])))
pick = lambda docs, k: [d for dom in ("CL", "IT") for d in [x for x in docs if x["domain"] == dom][:k]]
longest = lambda docs, dom: max([x for x in docs if x["domain"] == dom], key=lambda x: len(x["text"]))
D_SMOKE = dict(D, train=pick(D["train"], 2) + [longest(D["train"], dm) for dm in ("CL", "IT")], dev=pick(D["dev"], 1))
print("longest training documents (sentences):", [len(x["text"]) for x in D_SMOKE["train"][-2:]])

In [ ]:
# 6 · Metric matching + stray-label cleanup
import zipfile
from collections import Counter
M = rr.load_probs()
grp = {}
for n in sorted(M):
    grp.setdefault(n.split("_")[1], []).append(n)
print({g: len(ns) for g, ns in grp.items()})
L = LABELS

def ens(gs):
    return dict(dev=rr.combine([rr.combine([M[n]["dev"] for n in grp[g]]) for g in gs]),
                test=rr.combine([rr.combine([M[n]["test"] for n in grp[g]]) for g in gs]),
                tr=rr.mean_trans([M[n]["tr"] for g in gs for n in grp[g]]))

def mk_guard(extra=()):
    return dict(forbid=tuple(rr.GUARD["forbid"]) + tuple(extra), min_block=rr.GUARD["min_block"])

def decode_docs(b, split, docs, lam=1.0, k=0):
    out = {}
    for d in docs:
        p = b[split][d["id"]]; drop = []
        path = rr.decode_doc(p, L, b["tr"], lam, mk_guard())
        for _ in range(4):                                    # remove roles predicted for < k sentences, re-decode
            c = Counter(path); rare = [L[i] for i, n in c.items() if n < k and L[i] not in drop]
            if not rare or len(c) - len(rare) < 1: break
            drop += rare; path = rr.decode_doc(p, L, b["tr"], lam, mk_guard(drop))
        out[d["id"]] = [L[i] for i in path]
    return out

def f1s(gold, pred, labels_fixed=None):
    tp, fp, fn = Counter(), Counter(), Counter()
    for g, p in zip(gold, pred):
        if g == p: tp[g] += 1
        else: fp[p] += 1; fn[g] += 1
    used = labels_fixed or [l for l in L if tp[l] + fn[l] or fp[l]]
    f = []
    for l in used:
        P_ = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0; R_ = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        f.append(2 * P_ * R_ / (P_ + R_) if P_ + R_ else 0.0)
    return sum(f) / max(1, len(f))

def metrics(pred, docs):
    G = [l for d in docs for l in d["labels"]]; Pp = [l for d in docs for l in pred[d["id"]]]
    dom = [f1s([l for d in docs if d["domain"] == x for l in d["labels"]], [l for d in docs if d["domain"] == x for l in pred[d["id"]]])
           for x in ("CL", "IT")]
    return dict(pooled=f1s(G, Pp), all13=f1s(G, Pp, L), domains=sum(dom) / 2,
                documents=sum(f1s(d["labels"], pred[d["id"]]) for d in docs) / len(docs))

LB = {("v3a",): (0.4627, 1.0), ("v3b",): (0.4893, 1.0), ("v3c",): (0.4166, 1.0), ("v4a",): (0.4721, 1.0),
      ("v4b",): (0.4574, 1.0), ("v3a", "v3b"): (0.4709, 1.5)}
rows = []
for gs, (lb, lam) in LB.items():
    if all(g in grp for g in gs):
        m = metrics(decode_docs(ens(list(gs)), "dev", D["dev"], lam), D["dev"])
        rows.append((gs, lb, m)); print("+".join(gs), "LB", lb, {k: round(v, 4) for k, v in m.items()})
print("\n==== which dev metric reproduces the leaderboard? (mean |LB - dev|, lower = better match) ====")
err = {k: sum(abs(lb - m[k]) for _, lb, m in rows) / len(rows) for k in rows[0][2]}
for k, v in sorted(err.items(), key=lambda kv: kv[1]): print(f"   {k:<10} {v:.4f}")
METRIC = min(err, key=err.get)
print("BEST MATCH:", METRIC)

print("\n==== stray-label cleanup on dev (v3b) ====")
B = ens(["v3b"]) if "v3b" in grp else ens([max(grp)])
res = {}
for k in (0, 2, 3, 4, 6, 8, 12):
    m = metrics(decode_docs(B, "dev", D["dev"], 1.0, k), D["dev"]); res[k] = m
    print(f"k={k:<3}", {kk: round(v, 4) for kk, v in m.items()})
KA = max(res, key=lambda k: res[k][METRIC])
KB = max(res, key=lambda k: res[k]["pooled"])
print(f"chosen k for the matching metric ({METRIC}): {KA} | for the pooled metric: {KB}")

def write(pred, name):
    for d in D["test"]:
        assert len(pred[d["id"]]) == len(d["text"]) and all(l in L for l in pred[d["id"]])
    od = os.path.join(OUT, name); os.makedirs(od, exist_ok=True)
    pj = os.path.join(od, "prediction.json"); json.dump(pred, open(pj, "w"), ensure_ascii=False)
    zp = os.path.join(od, f"submission_{name}.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, arcname="prediction.json")
    return zp

ZA = write(decode_docs(B, "test", D["test"], 1.0, KA), f"p3_v8_A_k{KA}")
ZB = write(decode_docs(B, "test", D["test"], 1.0, max(KB, 3) if KB != KA else 4), f"p3_v8_B")
print("\nSUBMIT FIRST:", ZA, "\nTHEN       :", ZB)

In [ ]:
# 7 · (nothing to train)

In [ ]:
# 8 · (see cell 6)